In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error

In [2]:
df = pd.read_excel('Final_df.xlsx')
df['Datetime'] = df['Date'] + ' ' + df['Time']
df['Datetime'] = pd.to_datetime(df['Datetime'],dayfirst=True)
df.drop(['Date','Time'],axis=1,inplace=True)
df.set_index('Datetime',inplace=True)


In [3]:
df['hour'] = df.index.hour
df['day'] = df.index.day
df['month'] = df.index.month
df['year'] = df.index.year

In [4]:
df

,Temperature (°C),Precipitation (%),Holiday (0/1),Day of the Week (1-7),Load Demand (MW),hour,day,month,year
Datetime,,,,,,,,,
2021-09-14 00:00:00,28.9,2.0,0,2,7565.12,0,14,9,2021
2021-09-14 01:00:00,28.9,2.0,0,2,7111.65,1,14,9,2021
2021-09-14 02:00:00,28.9,2.0,0,2,6745.40,2,14,9,2021
2021-09-14 03:00:00,28.9,2.0,0,2,6419.14,3,14,9,2021
2021-09-14 04:00:00,28.9,2.0,0,2,6199.00,4,14,9,2021
...,...,...,...,...,...,...,...,...,...
2024-09-08 19:00:00,27.0,14.0,1,7,9618.32,19,8,9,2024
2024-09-08 20:00:00,27.0,14.0,1,7,9629.76,20,8,9,2024
2024-09-08 21:00:00,27.0,14.0,1,7,9693.51,21,8,9,2024


In [5]:
model = keras.Sequential([
    keras.layers.Dense(50,input_shape=[8],activation='relu'),
    keras.layers.Dense(100,activation='relu'),
    keras.layers.Dense(100,activation='relu'),
    keras.layers.Dense(1)
])

C:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
model.compile(optimizer='adam',loss='mae')

In [7]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta = 0.001,
    restore_best_weights=True
)

In [8]:
X = df.drop(['Load Demand (MW)'],axis=1)
y = df['Load Demand (MW)']
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [9]:
history = model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    batch_size=512,
    epochs=500,
    callbacks=[early_stopping],
    verbose=0
)

In [10]:
history_df = pd.DataFrame(history.history)

In [11]:
history_df

,loss,val_loss
0,6415.594238,3646.026611
1,2410.252930,2223.959473
2,2215.412109,2189.550781
3,2207.386719,2186.381348
4,2205.592285,2183.431152
...,...,...
356,798.934326,796.073914
357,800.713989,796.965210
358,806.279602,817.438904
359,801.761780,812.242981


In [12]:
y_pred = model.predict(X_test)

195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [13]:
y_true = y_test

In [14]:
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = mse ** 0.5
print(f'MAE: {mae}')
print(f'RMSE: {rmse}')

MAE: 794.471583969933
RMSE: 1065.9659387083984


In [16]:
from sklearn.metrics import r2_score

In [17]:
r2_score(y_true,y_pred)

0.8424928644089282

In [19]:
model.save('deep_learning_model.keras')